# Hamilton Heater Shaker quickstart

The Hamilton Heater Shaker (HHS) heats and shakes the plate on it, and locks the plate in place. PyLabRobot drives it through what it is plugged into: a STAR's TCC port, or anything else that sends its firmware commands.

| Property | Value |
|---|---|
| Communication | Firmware commands to module `T<index>`, through the STAR it is plugged into |
| Variants | Cat. no. 199027, 199033, 199034 |
| Temperature | Heating only, up to 105 °C |
| Speed | 20 – 2000 rpm |

```{warning}
This driver has not been run on a device. If you run it on one, please report how it goes on the [forum](https://discuss.pylabrobot.org).
```

```{device-card} hamilton-heater-shaker
```

## How it communicates

Each heater shaker plugged into a STAR is a firmware module of that STAR: `T1` on the first TCC port, `T2` on the second. Its commands go over the STAR's own USB link, so the STAR is set up first. `HamiltonHeaterShaker` needs only an object that sends those commands; a STAR's driver is one.

## Physical setup

1. Plug the heater shaker into a TCC port at the back of the STAR. The port's number is the heater shaker's index.
2. Fix the heater shaker on the deck and measure where its front left corner is.
3. Keep the plate off it until the plate lock has been initialized by `setup()`.

## Set up the STAR

The heater shaker's commands go through the STAR, so set it up first.

In [ ]:
from pylabrobot.hamilton.star import STARlet

star = STARlet()
await star.setup()

## Put the heater shaker on the deck

Its plate holder is a resource in the STAR's resource server, made from the `hamilton_heater_shaker` model. Give it the heater shaker's name, and put it where the heater shaker stands on the deck. This is done once; the resource server keeps it.

In [ ]:
from pylabrobot.resources.coordinate import Coordinate

await star.resource_server.create(
  "hamilton_heater_shaker",
  "hhs_1",
  parent=star.deck,
  location=Coordinate(x=72.24, y=242.28, z=107.44),  # Replace with where yours stands.
)

## Create the heater shaker

Give it the same name, the TCC port it is plugged into, and the STAR's driver.

In [ ]:
from pylabrobot.hamilton.heater_shaker import HamiltonHeaterShaker

hs = HamiltonHeaterShaker(
  name="hhs_1",
  index=1,
  interface=star.driver,
  resource_server=star.resource_server,
)

## Setup

`setup()` finds the heater shaker's plate holder, then initializes its plate lock and homes its shaker drive. It fails, before sending anything, if the plate holder is not in the resource server.

In [ ]:
await hs.setup()
hs.resource

## Lock and unlock the plate

`hs.resource` is where a plate is put on it, by hand or with the iSWAP. Lock the plate once it is there.

In [ ]:
await hs.lock_plate()

Unlock it to take the plate off.

In [ ]:
await hs.unlock_plate()

## Shake

`shake` locks the plate and shakes it, in rpm. With a duration, it stops and unlocks the plate after that many seconds.

In [ ]:
await hs.shake(speed=800, duration=10)

Without a duration it keeps shaking until it is told to stop.

In [ ]:
await hs.shake(speed=800)
await hs.get_is_shaking()

Stop shaking.

In [ ]:
await hs.stop_shaking()

## Heat

Set a temperature in Celsius and wait for it. The heater shaker cannot cool: below its current temperature, `set_temperature` refuses unless `passive=True` lets it cool down by itself.

In [ ]:
await hs.set_temperature(37)
await hs.wait_for_temperature()

Read the temperature in its middle and at its edge.

In [ ]:
await hs.get_temperature(), await hs.get_edge_temperature()

Turn the heating off.

In [ ]:
await hs.deactivate()

## On a heater shaker server

Where the STAR is served, heater shakers can be served too. The server sends their commands to the STAR through its STAR server:

```sh
python -m pylabrobot.hamilton.heater_shaker.server \
  --star-server http://lab-star:8081 --resource-server http://lab-star:8080 \
  --heater-shaker hhs_1:1 --port 8082
```

Anywhere else, the same methods are calls to that server; only the constructor changes. A long call (a shake, a wait for temperature, a hold) streams its progress, and a shake finishes on the server, stopping and unlocking the plate, even if the client goes away.

In [ ]:
served_hs = HamiltonHeaterShaker(
  name="hhs_1",
  heater_shaker_server="http://lab-star:8082",
  resource_server="http://lab-star:8080",
)

## Stop

Turn the heating off and stop shaking, then stop the STAR.

In [ ]:
await hs.stop()
await star.stop()